In [1]:
!pip install mlflow dagshub optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import numpy as np
import pandas as pd
import utils_data_clean_preprocessed
import mlflow
import optuna
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler, PowerTransformer, OrdinalEncoder
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor,StackingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [5]:
import dagshub
dagshub.init(repo_owner='Iamkartikey44', repo_name='Food_Delivery_Time_Prediction', mlflow=True)

Initialized MLflow to track repo "Iamkartikey44/Food_Delivery_Time_Prediction"

Repository Iamkartikey44/Food_Delivery_Time_Prediction initialized!

In [6]:
#Mlflow Experiment
mlflow.set_experiment("Exp4 - XGBoost_LGBM_RF HP")

2026/03/08 17:09:22 INFO mlflow.tracking.fluent: Experiment with name 'Exp4 - XGBoost_LGBM_RF HP' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1449875d257643b9ac5ee2ed2142acc5', creation_time=1772989762543, experiment_id='4', last_update_time=1772989762543, lifecycle_stage='active', name='Exp4 - XGBoost_LGBM_RF HP', tags={}, workspace='default'>

In [7]:
from sklearn import set_config
set_config(transform_output='pandas')

In [8]:
#Load the clean data
df = pd.read_csv("swiggy.csv")

In [9]:
#Load the clean data
df1 = utils_data_clean_preprocessed.perform_data_cleaning(df)

In [10]:
df1.head()

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
0,37.0,4.9,sunny,high,2,snack,motorcycle,0.0,no,urban,24,1,15.0,morning,0.0,short
1,34.0,4.5,stormy,jam,2,snack,scooter,1.0,no,metropolitian,33,0,5.0,evening,0.0,short
2,23.0,4.4,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,26,1,15.0,morning,0.0,short
3,38.0,4.7,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,21,0,10.0,evening,0.0,short
4,32.0,4.6,cloudy,high,1,snack,scooter,1.0,no,metropolitian,30,1,15.0,afternoon,0.0,short


In [11]:
df1.columns

Index(['age', 'ratings', 'weather', 'traffic', 'vehicle_condition',
       'type_of_order', 'type_of_vehicle', 'multiple_deliveries', 'festival',
       'city_type', 'time_taken', 'is_weekend', 'pickup_time_minutes',
       'order_time_of_day', 'distance', 'distance_type'],
      dtype='object')

In [12]:
df1.isna().sum()

,0
age,0
ratings,0
weather,0
traffic,0
vehicle_condition,0
type_of_order,0
type_of_vehicle,0
multiple_deliveries,0
festival,0
city_type,0


In [13]:
df1.duplicated().sum()

np.int64(24)

In [14]:
temp_df= df1.copy()

In [15]:
# split into X and y

X = temp_df.drop(columns='time_taken')
y = temp_df['time_taken']

In [16]:
#Train Test Split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [17]:
print(f"The size of train data is: {X_train.shape}")
print(f"The shape of test data is: {X_test.shape}")

The size of train data is: (30156, 15)
The shape of test data is: (7539, 15)


In [18]:
#Transform Target Column
pt = PowerTransformer()
y_train_pt = pt.fit_transform(y_train.values.reshape(-1,1))
y_test_pt = pt.transform(y_test.values.reshape(-1,1))

In [19]:
num_cols = ["age","ratings","pickup_time_minutes","distance"]

nominal_cat_cols = ['weather',
                    'type_of_order',
                    'type_of_vehicle',
                    "festival",
                    "city_type",
                    "is_weekend",
                    "order_time_of_day"]

ordinal_cat_cols = ["traffic","distance_type"]

In [20]:
# generate order for ordinal encoding
traffic_order = ["low","medium","high","jam"]
distance_type_order = ["short","medium","long","very_long"]

In [21]:
# unique categories the ordinal columns
for col in ordinal_cat_cols:
    print(col,X_train[col].unique())

traffic ['jam' 'medium' 'high' 'low']
distance_type ['short']
Categories (4, object): ['short' < 'medium' < 'long' < 'very_long']


In [22]:
#Build a preprocessor
preprocessor = ColumnTransformer(transformers=[
    ("scale",MinMaxScaler(),num_cols),
    ('nominal_encode',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False),nominal_cat_cols),
    ("ordinal_encode",OrdinalEncoder(categories=[traffic_order,distance_type_order],encoded_missing_value=-999,handle_unknown='use_encoded_value',unknown_value=-1),ordinal_cat_cols)
],remainder='passthrough',n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False)

In [ ]:
preprocessor

ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                  remainder='passthrough',
                  transformers=[('scale', MinMaxScaler(),
                                 ['age', 'ratings', 'pickup_time_minutes',
                                  'distance']),
                                ('nominal_encode',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 ['weather', 'type_of_order', 'type_of_vehicle',
                                  'festival', 'city_type', 'is_weekend',
                                  'order_time_of_day']),
                                ('ordinal_encode',
                                 OrdinalEncoder(categories=[['low', 'medium',
                                                             'high', 'jam'],
                                                            ['short', 'medium',
                                                             'long',
                                                             'very_long']],
                                                encoded_missing_value=-999,
                                                handle_unknown='use_encoded_value',
                                                unknown_value=-1),
                                 ['traffic', 'distance_type'])],
                  verbose_feature_names_out=False)

In [23]:
# build the pipeline

processing_pipeline = Pipeline(steps=[("preprocess",preprocessor)])
processing_pipeline

Pipeline(steps=[('preprocess',
                 ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                                   remainder='passthrough',
                                   transformers=[('scale', MinMaxScaler(),
                                                  ['age', 'ratings',
                                                   'pickup_time_minutes',
                                                   'distance']),
                                                 ('nominal_encode',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['weather', 'type_of_order',
                                                   'type_of_vehicle',
                                                   'festival', 'city_type',
                                                   'is_weekend',
                                                   'order_time_of_day']),
                                                 ('ordinal_encode',
                                                  OrdinalEncoder(categories=[['low',
                                                                              'medium',
                                                                              'high',
                                                                              'jam'],
                                                                             ['short',
                                                                              'medium',
                                                                              'long',
                                                                              'very_long']],
                                                                 encoded_missing_value=-999,
                                                                 handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['traffic',
                                                   'distance_type'])],
                                   verbose_feature_names_out=False))])

In [24]:
X_train_trans = processing_pipeline.fit_transform(X_train)
X_test_trans = processing_pipeline.transform(X_test)

In [26]:
def objective(trial):

    with mlflow.start_run(nested=True):

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 10, 500),
            "max_depth": trial.suggest_int("max_depth", 1, 30),

            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.8),

            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),

            "min_child_weight": trial.suggest_int("min_child_weight", 5, 20),

            "gamma": trial.suggest_float("gamma", 0, 10),

            "reg_lambda": trial.suggest_float("reg_lambda", 0, 100),
            "reg_alpha": trial.suggest_float("reg_alpha", 0, 100),

            "random_state": 42,
            "n_jobs": -1
        }

        mlflow.log_params(params)

        xgb_reg = XGBRegressor(**params)

        model = TransformedTargetRegressor(
            regressor=xgb_reg,
            transformer=pt
        )

        model.fit(X_train_trans, y_train)

        y_pred_train = model.predict(X_train_trans)
        y_pred_test = model.predict(X_test_trans)

        # cross validation
        cv_score = cross_val_score(
            model,
            X_train_trans,
            y_train,
            cv=5,
            scoring="neg_mean_absolute_error",
            n_jobs=1
        )

        mean_score = -cv_score.mean()

        mlflow.log_metric("cv_score", mean_score)

        return mean_score

In [27]:
study = optuna.create_study(direction='minimize')

with mlflow.start_run(run_name='best_model'):
  study.optimize(objective,n_trials=40,n_jobs=-1,show_progress_bar=True)
  mlflow.log_params(study.best_params)
  mlflow.log_metric("best_score",study.best_value)
  best_xgb = XGBRegressor(**study.best_params)
  best_xgb.fit(X_train_trans,y_train_pt.values.ravel())
  y_pred_train = best_xgb.predict(X_train_trans)
  y_pred_test = best_xgb.predict(X_test_trans)
  # get the actual predictions values
  y_pred_train_org = pt.inverse_transform(y_pred_train.reshape(-1,1))
  y_pred_test_org = pt.inverse_transform(y_pred_test.reshape(-1,1))

  model = TransformedTargetRegressor(regressor=best_xgb,transformer=pt)
  scores = cross_val_score(model,X_train_trans,y_train,scoring='neg_mean_squared_error',cv=5,n_jobs=-1)
  #Log Metrics
  mlflow.log_metric("Training Error",mean_absolute_error(y_train,y_pred_train_org))
  mlflow.log_metric("Testing Error",mean_absolute_error(y_test,y_pred_test_org))
  mlflow.log_metric("Training R2",r2_score(y_train,y_pred_train_org))
  mlflow.log_metric("Testing R2",r2_score(y_test,y_pred_test_org))
  mlflow.log_metric("Cross Val Score",scores.mean())

  mlflow.sklearn.log_model(best_xgb,artifact_path="model")


[I 2026-03-08 17:14:04,505] A new study created in memory with name: no-name-118d0da7-1c18-4239-a5ec-9fabd8ab4a9e


  0%|          | 0/50 [00:00<?, ?it/s]

🏃 View run big-grouse-634 at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4/runs/c247cea803a54a0eb196145726908707
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4
[I 2026-03-08 17:14:08,738] Trial 1 finished with value: 6.605313110351562 and parameters: {'n_estimators': 23, 'max_depth': 3, 'learning_rate': 0.015106198690459649, 'subsample': 0.6474770821882603, 'colsample_bytree': 0.9046584716340274, 'min_child_weight': 19, 'gamma': 4.731630260422124, 'reg_lambda': 20.40331193686622, 'reg_alpha': 3.8583218570648503}. Best is trial 1 with value: 6.605313110351562.
🏃 View run flawless-cow-541 at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4/runs/6c513d574b35425bbaef8d43ce84de4c
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4
[I 2026-03-08 17:14:12,591] Trial 0 finished with value: 4.30161

2026/03/08 17:18:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/08 17:18:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run best_model at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4/runs/d77f94d166234d1fa0501bcdd29ff5e4
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4


In [36]:
study.best_params

{'n_estimators': 415,
 'max_depth': 20,
 'learning_rate': 0.29263456258035475,
 'subsample': 0.8759669065136347,
 'colsample_bytree': 0.7213914971809194,
 'min_child_weight': 17,
 'gamma': 1.0059958323704052,
 'reg_lambda': 37.94112216359326,
 'reg_alpha': 0.5695261402488025}

In [29]:
study.best_value

3.8295311450958254

In [ ]:
study.best_value

3.8787633895874025

In [35]:
# optimization history plot
optuna.visualization.plot_optimization_history(study)

In [37]:
# plot hyperparameter importance plot
optuna.visualization.plot_param_importances(study)

In [38]:
# slice plot
optuna.visualization.plot_slice(study)

# LGBM HP Tuning

In [39]:
def objective_lgbm(trial):
    with mlflow.start_run(nested=True):
        params = {
            "n_estimators": trial.suggest_int("n_estimators",10,500),
            "max_depth": trial.suggest_int("max_depth",1,40),
            "learning_rate": trial.suggest_float("learning_rate",0.1,0.8),
            "subsample": trial.suggest_float("subsample",0.5,1),
            "min_child_weight": trial.suggest_int("min_child_weight",5,20),
            "min_split_gain": trial.suggest_float("min_split_gain",0,10),
            "reg_lambda": trial.suggest_float("reg_lambda",0,100),
            "random_state": 42,
            "n_jobs": -1,
        }

        # log model parameters
        mlflow.log_params(params)

        xgb_reg = LGBMRegressor(**params)
        model = TransformedTargetRegressor(regressor=xgb_reg,transformer=pt)

        # train the model
        model.fit(X_train_trans,y_train)

        # get the predictions
        y_pred_train = model.predict(X_train_trans)
        y_pred_test = model.predict(X_test_trans)


        # perform cross validation
        cv_score = cross_val_score(model,
                                X_train_trans,
                                y_train,
                                cv=5,
                                scoring="neg_mean_absolute_error",
                                n_jobs=-1)

        # mean score
        mean_score = -(cv_score.mean())
        # log avg cross val error
        mlflow.log_metric("cross_val_error",mean_score)

        return mean_score

In [40]:
# create optuna study
study_lgbm = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="best_model"):
    # optimize the objective function
    study_lgbm.optimize(objective_lgbm,n_trials=30,n_jobs=-1,show_progress_bar=True)

    # log the best parameters
    mlflow.log_params(study_lgbm.best_params)

    # log the best score
    mlflow.log_metric("best_score",study_lgbm.best_value)

    # train the model on best parameters
    best_lgbm = LGBMRegressor(**study_lgbm.best_params)

    best_lgbm.fit(X_train_trans,y_train_pt.values.ravel())

    # get the predictions
    y_pred_train = best_lgbm.predict(X_train_trans)
    y_pred_test = best_lgbm.predict(X_test_trans)

    # get the actual predictions values
    y_pred_train_org = pt.inverse_transform(y_pred_train.reshape(-1,1))
    y_pred_test_org = pt.inverse_transform(y_pred_test.reshape(-1,1))


    # perform cross validation
    model = TransformedTargetRegressor(regressor=best_lgbm,
                                        transformer=pt)


    scores = cross_val_score(model,
                         X_train_trans,
                         y_train,
                         scoring="neg_mean_absolute_error",
                         cv=5,n_jobs=-1)

    # log metrics
    mlflow.log_metric("training_error",mean_absolute_error(y_train,y_pred_train_org))
    mlflow.log_metric("test_error",mean_absolute_error(y_test,y_pred_test_org))
    mlflow.log_metric("training_r2",r2_score(y_train,y_pred_train_org))
    mlflow.log_metric("test_r2",r2_score(y_test,y_pred_test_org))
    mlflow.log_metric("cross_val",- scores.mean())

    # log the best model
    mlflow.sklearn.log_model(best_lgbm,artifact_path="model")

[I 2026-03-08 17:22:37,868] A new study created in memory with name: no-name-3dfe9b3d-ba21-4456-920a-2e0ed8968e7e


  0%|          | 0/30 [00:00<?, ?it/s]

🏃 View run thoughtful-fox-418 at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4/runs/86902510bd03499f9d973192d82534dd
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4
[I 2026-03-08 17:22:47,368] Trial 1 finished with value: 4.102902325912611 and parameters: {'n_estimators': 28, 'max_depth': 5, 'learning_rate': 0.5422272190410119, 'subsample': 0.9893259576137438, 'min_child_weight': 15, 'min_split_gain': 1.2514371731536522, 'reg_lambda': 28.866438725329367}. Best is trial 1 with value: 4.102902325912611.
🏃 View run clumsy-fox-416 at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4/runs/40579f53575e4b46949e2f1fc1b645e5
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4
[I 2026-03-08 17:22:47,579] Trial 0 finished with value: 4.212433577341244 and parameters: {'n_estimators': 254, 'max_depth': 

2026/03/08 17:26:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/08 17:26:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run best_model at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4/runs/2625240f7ef041889a70c3cbe6810794
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4


In [42]:
study_lgbm.best_params

{'n_estimators': 398,
 'max_depth': 30,
 'learning_rate': 0.17723976615372525,
 'subsample': 0.6668154532640596,
 'min_child_weight': 13,
 'min_split_gain': 0.00042029710843746043,
 'reg_lambda': 75.85847527149905}

In [ ]:
study_lgbm.best_params

{'n_estimators': 152,
 'max_depth': 19,
 'learning_rate': 0.7444003876852524,
 'subsample': 0.5284402935634638,
 'min_child_weight': 9,
 'min_split_gain': 0.014263522429579317,
 'reg_lambda': 55.54275117208787}

In [41]:
study_lgbm.best_value

3.8409433389512

In [ ]:
study_lgbm.best_value

3.9072908835002678

In [43]:
# optimization history plot
optuna.visualization.plot_optimization_history(study_lgbm)

In [44]:
# plot hyperparameter importance plot
optuna.visualization.plot_param_importances(study_lgbm)

In [45]:
# slice plot
optuna.visualization.plot_slice(study_lgbm)

# RF HP Tuning

In [63]:
def objective_rf(trial):
    with mlflow.start_run(nested=True):
        params = {
                  "n_estimators": trial.suggest_int("n_estimators", 100, 800),

                  "max_depth": trial.suggest_int("max_depth", 3, 20),

                  "max_features": trial.suggest_categorical(
                      "max_features",
                      ["sqrt", "log2", None]
                  ),

                  "min_samples_split": trial.suggest_int(
                      "min_samples_split",
                      2,
                      10
                  ),

                  "min_samples_leaf": trial.suggest_int(
                      "min_samples_leaf",
                      1,
                      10
                  ),

                  "max_samples": trial.suggest_float(
                      "max_samples",
                      0.7,
                      1.0
                  ),



                  "random_state": 42,
                  "n_jobs": -1,
              }

        # log model parameters
        mlflow.log_params(params)

        # build the model
        rf = RandomForestRegressor(**params)
        model = TransformedTargetRegressor(regressor=rf,transformer=pt)

        # train the model
        model.fit(X_train_trans,y_train)

        # get the predictions
        y_pred_train = model.predict(X_train_trans)
        y_pred_test = model.predict(X_test_trans)


        # perform cross validation
        cv_score = cross_val_score(model,
                                X_train_trans,
                                y_train,
                                cv=5,
                                scoring="neg_mean_absolute_error",
                                n_jobs=-1)

        # mean score
        mean_score = -(cv_score.mean())

        # log avg cross val error
        mlflow.log_metric("cross_val_error",mean_score)

        return mean_score

In [64]:
# create optuna study
study_rf = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="best_model"):
    # optimize the objective function
    study_rf.optimize(objective_rf,n_trials=20,n_jobs=-1,show_progress_bar=True)

    # log the best parameters
    mlflow.log_params(study_rf.best_params)

    # log the best score
    mlflow.log_metric("best_score",study_rf.best_value)

    # train the model on best parameters
    best_rf = RandomForestRegressor(**study_rf.best_params)

    best_rf.fit(X_train_trans,y_train_pt.values.ravel())

    # get the predictions
    y_pred_train = best_rf.predict(X_train_trans)
    y_pred_test = best_rf.predict(X_test_trans)

    # get the actual predictions values
    y_pred_train_org = pt.inverse_transform(y_pred_train.reshape(-1,1))
    y_pred_test_org = pt.inverse_transform(y_pred_test.reshape(-1,1))


    # perform cross validation
    model = TransformedTargetRegressor(regressor=best_rf,transformer=pt)


    scores = cross_val_score(model,X_train_trans,y_train,scoring="neg_mean_absolute_error",cv=5,n_jobs=-1)

    # log metrics
    mlflow.log_metric("training_error",mean_absolute_error(y_train,y_pred_train_org))
    mlflow.log_metric("test_error",mean_absolute_error(y_test,y_pred_test_org))
    mlflow.log_metric("training_r2",r2_score(y_train,y_pred_train_org))
    mlflow.log_metric("test_r2",r2_score(y_test,y_pred_test_org))
    mlflow.log_metric("cross_val",- scores.mean())

    # log the best model
    mlflow.sklearn.log_model(best_rf,artifact_path="model")

[I 2026-03-08 18:11:36,001] A new study created in memory with name: no-name-3edc6fe1-07ed-499d-bfbd-dbf30d387b5c


  0%|          | 0/20 [00:00<?, ?it/s]

🏃 View run delicate-fly-720 at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4/runs/e601e7e77ada40a888d120a5810ceaa9
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4
[I 2026-03-08 18:12:31,034] Trial 0 finished with value: 5.065500006708376 and parameters: {'n_estimators': 681, 'max_depth': 4, 'max_features': 'sqrt', 'min_samples_split': 8, 'min_samples_leaf': 2, 'max_samples': 0.7208174358150577}. Best is trial 0 with value: 5.065500006708376.
🏃 View run likeable-owl-295 at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4/runs/85631e4308fe4353a514f2e0b2211ea4
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4
[I 2026-03-08 18:12:31,986] Trial 1 finished with value: 4.184766482267713 and parameters: {'n_estimators': 212, 'max_depth': 7, 'max_features': None, 'min_samples_split': 10, 'min_samp

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning:

A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.



🏃 View run respected-chimp-850 at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4/runs/c0748a3a57804e94a39abad221eb76f2
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4
[I 2026-03-08 18:14:56,839] Trial 4 finished with value: 5.2956144060849395 and parameters: {'n_estimators': 747, 'max_depth': 3, 'max_features': None, 'min_samples_split': 4, 'min_samples_leaf': 6, 'max_samples': 0.9664911660201392}. Best is trial 3 with value: 4.017305838350035.
🏃 View run melodic-crab-647 at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4/runs/81fc7b4d50b94ea190db3681783517b0
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4
[I 2026-03-08 18:15:03,814] Trial 5 finished with value: 3.909053712144041 and parameters: {'n_estimators': 574, 'max_depth': 15, 'max_features': 'log2', 'min_samples_split': 2, 'min_

...

🏃 View run mysterious-elk-79 at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4/runs/9278ca3ad5044f51b1bfb9a96381518c
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4
[I 2026-03-08 18:19:09,497] Trial 13 finished with value: 3.9410889860153793 and parameters: {'n_estimators': 444, 'max_depth': 20, 'max_features': 'log2', 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_samples': 0.7032052919516979}. Best is trial 5 with value: 3.909053712144041.
🏃 View run unleashed-sponge-588 at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4/runs/b1a06a7eafeb4600b4d61849168630a3
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4
[I 2026-03-08 18:19:16,539] Trial 12 finished with value: 3.8936227482344514 and parameters: {'n_estimators': 427, 'max_depth': 19, 'max_features': 'log2', 'min_samples_split': 

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning:

A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.



🏃 View run awesome-bat-811 at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4/runs/b2ac2569e9b248bbb477d0634f5574f4
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4
[I 2026-03-08 18:20:52,966] Trial 14 finished with value: 3.884512455965713 and parameters: {'n_estimators': 378, 'max_depth': 20, 'max_features': 'log2', 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_samples': 0.7531529960526339}. Best is trial 14 with value: 3.884512455965713.
🏃 View run blushing-sloth-445 at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4/runs/091089b405f747629edf996271ba0dc9
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4
[I 2026-03-08 18:21:01,118] Trial 15 finished with value: 3.8971080553177573 and parameters: {'n_estimators': 399, 'max_depth': 17, 'max_features': 'log2', 'min_samples_split': 2, '

2026/03/08 18:28:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/08 18:29:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run best_model at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4/runs/2b7dd844a7574cb09ac8db9d7c387763
🧪 View experiment at: https://dagshub.com/Iamkartikey44/Food_Delivery_Time_Prediction.mlflow/#/experiments/4


In [65]:
study_rf.best_params #old

{'n_estimators': 539,
 'max_depth': 14,
 'max_features': None,
 'min_samples_split': 5,
 'min_samples_leaf': 5,
 'max_samples': 0.9188618214071786}

In [59]:
study_rf.best_params

{'n_estimators': 500,
 'max_depth': 14,
 'max_features': None,
 'min_samples_split': 9,
 'min_samples_leaf': 6,
 'max_samples': 0.9797814656591307}

In [67]:
study_rf.best_value

3.807369054223122

In [68]:
study_rf.best_value #OLD

3.807369054223122

In [61]:
# optimization history plot
optuna.visualization.plot_optimization_history(study_rf)

In [62]:
# plot hyperparameter importance plot
optuna.visualization.plot_param_importances(study_rf)

In [53]:
# slice plot
optuna.visualization.plot_slice(study_rf)